# Warehouse hazard triage from images: d1 decision models vs generative VLMs vs a frontier model

**Question:** for spotting safety hazards in warehouse images, how do LiquidAI's new **d1** decision models compare with ordinary **generative VLMs** (including d1-3B's own base model), a **small fine-tuned VLM**, and a **frontier model** (Claude Opus 5.5)? For each option: how accurate is it, how many hazards does it miss, how many false alarms does it raise, and what does it cost per 1,000 images?

**Setup**
- **Data:** [`Jsohal174/warehouse-safety-hazard-dataset`](https://huggingface.co/datasets/Jsohal174/warehouse-safety-hazard-dataset) (CC BY 4.0). It has 913 overhead warehouse images in 5 classes: spill, forklift violation, improper stacking, obstacle and safe. We use the official 186-image test set. From the 727 training images, 100 are held out (stratified, seed 42) for calibration and threshold choice, and the remaining 627 are used for fine-tuning.
- **Every model answers the same 5-way question** with the same class descriptions, and every image is resized to the same maximum size.
- **Decision models (zero-shot):** d1-3B and d1-omni-600M. They answer in one forward pass with a probability per class.
- **Generative VLMs (zero-shot):** LFM2.5-VL-3B (d1-3B's base model), Qwen3.5-2B and Qwen3.5-4B. The options are labelled A–E, and each model's next-token probabilities over A–E give its class probabilities.
- **Fine-tuned:** Qwen3.5-0.8B, LoRA-fine-tuned with Unsloth on the 627 training images to answer with the option letter.
- **Frontier (optional, paid):** Claude Opus 5.5 through the Anthropic API, with a structured-output label. It runs only if `ANTHROPIC_API_KEY` is set.

**TL;DR:** TODO(you) after running.

> **Caveat up front:** the images are **synthetic**. They are Blender renders made photorealistic, with hazards added using Google Gemini. Results show how the models compare on this benchmark, not how they'd do on real CCTV footage.

## Why this matters

Visual inspection is one of the most common business uses for vision models: safety audits, compliance checks, insurance photos and quality control. Teams usually choose between three options:
- an **API frontier model**, which is strong, but every image costs money and leaves your network;
- a **small open VLM** prompted to answer;
- a **fine-tuned small model**.

LiquidAI's **d1** models (released 7 October 2026) add a fourth option. They are "system-1" decision models that take an image and a typed question, and return calibrated probabilities in one forward pass, with no text generation. d1-3B was post-trained from LFM2.5-VL-3B, so comparing the two isolates what the decision post-training adds on the same backbone.

## Setup

In [ ]:
%pip install -q "unsloth==2026.10.3" "unsloth_zoo==2026.10.3" "transformers==5.17.0" "datasets==4.8.5" "anthropic==1.12.1" pillow scikit-learn scipy seaborn

In [ ]:
# Unsloth must be imported before transformers so its patches apply.
import unsloth
from unsloth import FastVisionModel, is_bfloat16_supported

import base64, gc, io, json, math, os, random, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch, transformers, datasets
from datasets import load_dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("unsloth", unsloth.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__,
      "| torch", torch.__version__)
assert torch.cuda.is_available(), "This notebook needs a GPU: Runtime -> Change runtime type -> T4 GPU"
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
DTYPE = torch.bfloat16 if is_bfloat16_supported() else torch.float16   # T4 has no bf16

## Config

In [ ]:
DATASET_ID = "Jsohal174/warehouse-safety-hazard-dataset"
CALIB_N = 100                 # training images held out for calibration and threshold choice
IMG_MAX = 768                 # longest side after resizing; the same images go to every model
SMOKE_N = None                # set e.g. 8 for a quick end-to-end check (separate cache, tiny fine-tune)

# Class descriptions follow the dataset card. Order fixes the option letters A-E.
CLASSES = {
    "spill": "Liquid spill on the floor (oil, water or chemicals)",
    "forklift_violation": "Unsafe forklift operation (overloading, tipping or speeding)",
    "improper_stacking": "Unstable or incorrectly stacked pallets or boxes",
    "obstacle": "Debris, equipment or objects blocking an aisle",
    "safe": "Clean, properly maintained area with no visible hazard",
}
CLASS_NAMES = list(CLASSES)
LETTERS = "ABCDE"
SAFE = CLASS_NAMES.index("safe")
HAZARD_RECALL_TARGET = 0.90   # threshold rule for "needs attention", applied on the calibration split only

D1_MODELS = {"d1-3B": "LiquidAI/d1-3B", "d1-omni-600M": "LiquidAI/d1-omni-600M"}
VLM_MODELS = {"LFM2.5-VL-3B": "LiquidAI/LFM2.5-VL-3B", "Qwen3.5-2B": "Qwen/Qwen3.5-2B", "Qwen3.5-4B": "Qwen/Qwen3.5-4B"}
FT_BASE = "unsloth/Qwen3.5-0.8B"
FT = dict(r=16, lora_alpha=16, lr=2e-4, epochs=2, batch=4, grad_accum=4)
GEN_CHECK_N = 30              # test images on which each generative VLM also writes its answer, as a sanity check

# Frontier model (optional). Prices are the Anthropic list prices per million tokens.
CLAUDE_MODEL = "claude-opus-5-5"
CLAUDE_USD_PER_MTOK = {"input": 4.00, "output": 20.00}
# Local cost model: an assumption, stated in the write-up. One image at a time on one T4.
T4_USD_PER_HOUR = 0.35

QUESTION = ("You are inspecting an overhead warehouse safety camera image. "
            "Which one option best describes what the image shows?")
CHOICE_Q = {"type": "choice", "instructions": QUESTION, "criteria": CLASSES}
LETTER_PROMPT = (QUESTION + "\n" + "\n".join(f"{L}. {d}" for L, d in zip(LETTERS, CLASSES.values()))
                 + "\nAnswer with the letter only.")

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/ML/2026-10-warehouse-hazards-d1-vlm")
else:
    BASE_DIR = Path("outputs")
CACHE_DIR = BASE_DIR / ("cache_smoke" if SMOKE_N else "cache")
CKPT_DIR, FIG_DIR = BASE_DIR / "checkpoints", Path("figures")
for d in (CACHE_DIR, CKPT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

## Data

In [ ]:
from sklearn.model_selection import train_test_split


def prep(img):
    img = img.convert("RGB")
    img.thumbnail((IMG_MAX, IMG_MAX))
    return img


def to_frame(ds, split):
    d = ds.remove_columns("image").to_pandas()
    d["uid"] = [f"{split}_{i}" for i in range(len(d))]
    d["label"] = d.category.map(CLASS_NAMES.index)
    # accept_also lists other labels the dataset author also accepts for ambiguous images
    d["accept"] = [{c} | set(str(a).split(",")) - {"nan", ""} for c, a in zip(d.category, d.accept_also)]
    return d[["uid", "category", "label", "accept", "description", "severity"]]


train_raw = load_dataset(DATASET_ID, split="train")
test_raw = load_dataset(DATASET_ID, split="test")
IMAGES = {}
for split, ds in [("train", train_raw), ("test", test_raw)]:
    for i, img in enumerate(tqdm(ds["image"], desc=f"resizing {split}")):
        IMAGES[f"{split}_{i}"] = prep(img)

train_all, test_df = to_frame(train_raw, "train"), to_frame(test_raw, "test")
train_df, calib_df = train_test_split(train_all, test_size=CALIB_N, stratify=train_all.category, random_state=SEED)
if SMOKE_N:
    train_df = train_df.groupby("category").head(8)
    calib_df, test_df = calib_df.sample(SMOKE_N, random_state=SEED), test_df.sample(SMOKE_N, random_state=SEED)
train_df, calib_df, test_df = (d.reset_index(drop=True) for d in (train_df, calib_df, test_df))
assert not set(train_df.uid) & set(calib_df.uid)

counts = pd.DataFrame({n: d.category.value_counts() for n, d in [("train", train_df), ("calib", calib_df), ("test", test_df)]})
print(f"train {len(train_df)} | calib {len(calib_df)} | test {len(test_df)} | images resized to <= {IMG_MAX}px")
display(counts)

fig, axes = plt.subplots(1, 5, figsize=(18, 3.6))
for ax, c in zip(axes, CLASS_NAMES):
    ax.imshow(IMAGES[test_df[test_df.category == c].uid.iloc[0]]); ax.set_title(c, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.savefig(FIG_DIR / "examples.png", dpi=120, bbox_inches="tight"); plt.show()

**Licence and provenance:** CC BY 4.0, by the dataset's author on Hugging Face. The images are synthetic: Blender renders from an overhead drone viewpoint, made photorealistic with Google Gemini, which also added the hazards. 25 test images list a second acceptable label (`accept_also`), so accuracy is reported both strictly and leniently. The `severity` field is not used: it mixes 7 inconsistent values, and some "safe" images are marked high.

**Published reference points** from the dataset card, on the same 186-image test set: Qwen3.5-2B zero-shot 67.2%, Qwen3.5-4B zero-shot 83.3%, and Qwen3.5-2B fine-tuned ("Hawkeye") 92.5–94.1%. Our prompt differs from theirs, so expect similar rather than identical numbers.

## Shared helpers

Every local model returns a probability per class. The **predicted class** is the most likely one, and **P(hazard)** is 1 − P(safe). Predictions are cached to disk as JSONL, so a rerun after a Colab disconnect resumes where it stopped.

In [ ]:
def load_cache(path):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        return {rec["uid"]: rec for rec in map(json.loads, f)}


def run_model(name, predict_fn, d, split):
    # predict_fn(image) -> dict with "probs" (one per class, CLASS_NAMES order) and optional extra fields.
    path = CACHE_DIR / f"{name}_{split}.jsonl"
    done = load_cache(path)
    todo = d[~d.uid.isin(done)]
    print(f"{name} [{split}]: {len(done)} cached, {len(todo)} to run")
    if not len(todo):
        return
    torch.cuda.reset_peak_memory_stats()
    predict_fn(IMAGES[todo.uid.iloc[0]])      # warm-up, not timed
    with path.open("a", encoding="utf-8") as f:
        for r in tqdm(todo.itertuples(), total=len(todo), desc=name):
            torch.cuda.synchronize(); t0 = time.perf_counter()
            out = predict_fn(IMAGES[r.uid])
            torch.cuda.synchronize()
            p = np.clip(np.asarray(out.pop("probs"), float), 0, None)
            f.write(json.dumps({"uid": r.uid, "probs": (p / p.sum()).tolist(),
                                "latency_s": time.perf_counter() - t0, **out}) + "\n")
    (CACHE_DIR / f"{name}_{split}_mem.json").write_text(json.dumps({"peak_gb": torch.cuda.max_memory_allocated() / 1e9}))


def free():
    # Call after dropping the last reference to a model.
    gc.collect(); torch.cuda.empty_cache()


SPLITS = [("test", test_df), ("calib", calib_df)]

## Decision models: d1-3B and d1-omni-600M (zero-shot)

The image is the whole state (`state=None`), and the question is a `choice` over the five class descriptions. The API is from the d1 model cards.

In [ ]:
from transformers import AutoModel

for name, model_id in D1_MODELS.items():
    if all(len(load_cache(CACHE_DIR / f"{name}_{s}.jsonl")) == len(d) for s, d in SPLITS):
        print(name, "already cached"); continue
    d1 = AutoModel.from_pretrained(model_id, trust_remote_code=True, dtype=DTYPE).to(DEVICE).eval()

    def predict(img):
        ans = d1.system_one(None, {"hazard": CHOICE_Q}, images=[img])["answers"]["hazard"]
        return {"probs": [float(ans["probabilities"][c]) for c in CLASS_NAMES]}

    with torch.inference_mode():
        for split, d in SPLITS:
            run_model(name, predict, d, split)
    d1 = None; free()

## Generative VLMs (zero-shot): LFM2.5-VL-3B, Qwen3.5-2B, Qwen3.5-4B

Each model sees the image and the options labelled A–E, and is asked for the letter only. Instead of parsing generated text, we read the model's next-token probabilities for the five letters. That gives class probabilities from one forward pass, so these models can be thresholded and calibration-checked like the decision models. As a sanity check, each model also *writes* its answer for the first `GEN_CHECK_N` test images, and we record whether it agrees with the top letter.

Qwen3.5 thinking is switched off (`enable_thinking=False`), so the answer letter is the next token.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor


def chat_inputs(processor, img, answer=None):
    messages = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": LETTER_PROMPT}]}]
    kwargs = dict(add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    try:
        return processor.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        return processor.apply_chat_template(messages, **kwargs)


def letter_ids(processor):
    tok = getattr(processor, "tokenizer", processor)
    ids = [tok.encode(L, add_special_tokens=False)[0] for L in LETTERS]
    assert len(set(ids)) == len(LETTERS), f"letters don't map to distinct tokens: {ids}"
    return ids, tok


def make_letter_scorer(model, processor, gen_uids=()):
    ids, tok = letter_ids(processor)

    @torch.inference_mode()
    def predict(img):
        inputs = chat_inputs(processor, img).to(model.device)
        logits = model(**inputs).logits[0, -1].float()
        probs = torch.softmax(logits[ids], -1).cpu().numpy()
        out = {"probs": probs.tolist(), "letter_mass": float(torch.softmax(logits, -1)[ids].sum())}
        if id(img) in gen_uids:
            gen = model.generate(**inputs, max_new_tokens=4, do_sample=False)
            out["generated"] = tok.decode(gen[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        return out
    return predict


GEN_IMAGE_IDS = {id(IMAGES[u]) for u in test_df.uid[:GEN_CHECK_N]}
for name, model_id in VLM_MODELS.items():
    if all(len(load_cache(CACHE_DIR / f"{name}_{s}.jsonl")) == len(d) for s, d in SPLITS):
        print(name, "already cached"); continue
    vlm = AutoModelForImageTextToText.from_pretrained(model_id, dtype=DTYPE).to(DEVICE).eval()
    proc = AutoProcessor.from_pretrained(model_id)
    predict = make_letter_scorer(vlm, proc, GEN_IMAGE_IDS)
    for split, d in SPLITS:
        run_model(name, predict, d, split)
    vlm = proc = predict = None; free()

## Fine-tuned: Qwen3.5-0.8B with Unsloth (vision LoRA)

The C-SSRS experiment fine-tuned decision heads with Unsloth's `FastDecisionModel`. In Unsloth 2026.10.3 its training items are built from the text state and questions only, with no images, so it can't train an image decision model yet. This arm instead uses Unsloth's standard vision fine-tune (`FastVisionModel` + `UnslothVisionDataCollator`, following [their docs](https://unsloth.ai/docs/basics/vision-fine-tuning)). The model learns to answer the same A–E prompt with the correct letter, and is then scored with the same next-token letter probabilities as the zero-shot VLMs.

The adapter is saved under `CKPT_DIR`, and training is skipped if a finished run exists.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FT_NAME = "Qwen3.5-0.8B + Unsloth"
ft_dir = CKPT_DIR / ("qwen0.8b_vision_lora" + ("_smoke" if SMOKE_N else ""))


def to_conversation(uid, label):
    return {"messages": [
        {"role": "user", "content": [{"type": "image", "image": IMAGES[uid]}, {"type": "text", "text": LETTER_PROMPT}]},
        {"role": "assistant", "content": [{"type": "text", "text": LETTERS[label]}]},
    ]}


if not (ft_dir / "done.json").exists():
    model, processor = FastVisionModel.from_pretrained(FT_BASE, load_in_4bit=False, use_gradient_checkpointing="unsloth")
    model = FastVisionModel.get_peft_model(model, finetune_vision_layers=True, finetune_language_layers=True,
                                           finetune_attention_modules=True, finetune_mlp_modules=True,
                                           r=FT["r"], lora_alpha=FT["lora_alpha"], lora_dropout=0, bias="none",
                                           random_state=SEED)
    FastVisionModel.for_training(model)
    train_data = [to_conversation(u, y) for u, y in zip(train_df.uid, train_df.label)]
    trainer = SFTTrainer(
        model=model, tokenizer=processor, data_collator=UnslothVisionDataCollator(model, processor),
        train_dataset=train_data,
        args=SFTConfig(
            per_device_train_batch_size=FT["batch"], gradient_accumulation_steps=FT["grad_accum"],
            num_train_epochs=1 if SMOKE_N else FT["epochs"], learning_rate=FT["lr"], warmup_steps=5,
            lr_scheduler_type="cosine", weight_decay=0.01, optim="adamw_8bit", logging_steps=5,
            bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(), seed=SEED,
            output_dir=str(ft_dir / "trainer"), report_to="none", save_strategy="no",
            remove_unused_columns=False, dataset_text_field="", dataset_kwargs={"skip_prepare_dataset": True},
            max_length=2048),
    )
    torch.cuda.reset_peak_memory_stats(); t0 = time.perf_counter()
    trainer.train()
    summary = {"train_minutes": round((time.perf_counter() - t0) / 60, 1),
               "train_peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 1),
               "history": [h for h in trainer.state.log_history if "loss" in h]}
    model.save_pretrained(str(ft_dir)); processor.save_pretrained(str(ft_dir))
    (ft_dir / "done.json").write_text(json.dumps(summary))
    print({k: v for k, v in summary.items() if k != "history"})
    model = processor = trainer = None; free()
else:
    print("already trained:", {k: v for k, v in json.loads((ft_dir / "done.json").read_text()).items() if k != "history"})

if not all(len(load_cache(CACHE_DIR / f"{FT_NAME}_{s}.jsonl")) == len(d) for s, d in SPLITS):
    model, processor = FastVisionModel.from_pretrained(str(ft_dir), load_in_4bit=False)
    FastVisionModel.for_inference(model)
    predict = make_letter_scorer(model, processor, GEN_IMAGE_IDS)
    for split, d in SPLITS:
        run_model(FT_NAME, predict, d, split)
    model = processor = predict = None; free()

## Frontier model: Claude Opus 5.5 (optional, paid)

This arm runs only if an Anthropic API key is available: a Colab secret named `ANTHROPIC_API_KEY`, or the environment variable. Otherwise it's skipped and the rest of the notebook still works. Each test image is sent as a JPEG, the same resized image as the local models get, together with the same question. Structured outputs constrain the reply to one of the five class names. Effort is `low`, since this is a simple classification. Server-side refusal fallback is enabled, and every response's token usage is recorded to compute cost.

Claude returns a label, not probabilities, so it has one operating point and no calibration check. Note that this sends the images to Anthropic's API.

In [ ]:
CLAUDE_NAME = "Claude Opus 5.5"
key = os.environ.get("ANTHROPIC_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        key = None
RUN_CLAUDE = bool(key)
print("Claude arm:", "on" if RUN_CLAUDE else "skipped (no ANTHROPIC_API_KEY)")


def jpeg_b64(img):
    buf = io.BytesIO(); img.save(buf, format="JPEG", quality=90)
    return base64.standard_b64encode(buf.getvalue()).decode("utf-8")


if RUN_CLAUDE:
    import anthropic
    client = anthropic.Anthropic(api_key=key)
    schema = {"type": "object", "properties": {"category": {"type": "string", "enum": CLASS_NAMES}},
              "required": ["category"], "additionalProperties": False}
    claude_prompt = QUESTION + "\n" + "\n".join(f"- {c}: {d}" for c, d in CLASSES.items())

    USE_FALLBACK = True   # server-side refusal fallback; switched off automatically if the API rejects it

    def ask_claude(img):
        global USE_FALLBACK
        request = dict(
            model=CLAUDE_MODEL, max_tokens=2000,
            output_config={"effort": "low", "format": {"type": "json_schema", "schema": schema}},
            messages=[{"role": "user", "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": jpeg_b64(img)}},
                {"type": "text", "text": claude_prompt}]}],
        )
        if USE_FALLBACK:
            try:
                r = client.beta.messages.create(**request, betas=["server-side-fallback-2026-07-01"], fallbacks="default")
            except anthropic.BadRequestError as e:
                print("Fallback not accepted, continuing without it:", e)
                USE_FALLBACK = False
        if not USE_FALLBACK:
            r = client.messages.create(**request)
        text = next((b.text for b in r.content if b.type == "text"), None)
        category = json.loads(text)["category"] if (text and r.stop_reason != "refusal") else None
        return {"category": category, "stop_reason": r.stop_reason, "served_by": r.model,
                "input_tokens": r.usage.input_tokens, "output_tokens": r.usage.output_tokens}

    path = CACHE_DIR / "claude_test.jsonl"
    done = load_cache(path)
    todo = test_df[~test_df.uid.isin(done)]
    if len(todo):
        first = ask_claude(IMAGES[todo.uid.iloc[0]])
        per_image = (first["input_tokens"] * CLAUDE_USD_PER_MTOK["input"]
                     + first["output_tokens"] * CLAUDE_USD_PER_MTOK["output"]) / 1e6
        print(f"First call: {first}. Estimated cost for {len(todo)} images: ${per_image * len(todo):.2f}")
    with path.open("a", encoding="utf-8") as f:
        for r in tqdm(todo.itertuples(), total=len(todo), desc=CLAUDE_NAME):
            t0 = time.perf_counter()
            for attempt in range(3):
                try:
                    out = ask_claude(IMAGES[r.uid]); break
                except (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.InternalServerError):
                    time.sleep(10 * (attempt + 1))
            else:
                print("giving up on", r.uid); continue
            f.write(json.dumps({"uid": r.uid, "latency_s": time.perf_counter() - t0, **out}) + "\n")

## Evaluation

All metrics are on the 186 test images.
- **Accuracy:** strict (the labelled class) and lenient (also accepting `accept_also`), with a 95% Wilson CI.
- **Macro-F1** and **per-class recall**.
- **Needs attention** (any hazard): hazard recall, false-alarm rate on the safe images, and precision, using P(hazard) = 1 − P(safe) ≥ 0.5. Claude uses its label.
- **ECE** on the top-class probability.
- **Median ms per image**, **peak GPU memory** and **cost per 1,000 images**:
  - local models: median latency × `T4_USD_PER_HOUR`, an assumption covering one image at a time on one T4;
  - Claude: measured tokens × list price.

In [ ]:
from scipy.stats import binomtest
from sklearn.metrics import confusion_matrix, f1_score

LOCAL = list(D1_MODELS) + list(VLM_MODELS) + [FT_NAME]
y_test = test_df.label.to_numpy()
hz_test = y_test != SAFE


def wilson(k, n, z=1.96):
    p, denom = k / n, 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return centre - half, centre + half


def ece(correct, conf, bins=10):
    correct, conf = np.asarray(correct, float), np.asarray(conf, float)
    which = np.clip(np.digitize(conf, np.linspace(0, 1, bins + 1)[1:-1]), 0, bins - 1)
    return sum(abs(correct[which == b].mean() - conf[which == b].mean()) * (which == b).mean()
               for b in range(bins) if (which == b).any())


def attention(flag, hz):
    tp, fp = int((flag & hz).sum()), int((flag & ~hz).sum())
    return {"hazards caught": f"{tp}/{int(hz.sum())}", "hazard recall": tp / hz.sum(),
            "false alarms": f"{fp}/{int((~hz).sum())}", "false-alarm rate": fp / (~hz).sum(),
            "precision": tp / max(tp + fp, 1), "flagged": int(flag.sum())}


def metrics(pred, p_hazard_flag, probs=None):
    n, k = len(y_test), int((pred == y_test).sum())
    lenient = np.mean([CLASS_NAMES[p] in acc if p >= 0 else False for p, acc in zip(pred, test_df.accept)])
    lo, hi = wilson(k, n)
    row = {"accuracy": k / n, "acc 95% CI": f"{lo:.1%}–{hi:.1%}", "lenient accuracy": lenient,
           "macro-F1": f1_score(y_test, pred, average="macro", labels=list(range(5)), zero_division=0),
           **attention(p_hazard_flag, hz_test),
           "ECE": ece(pred == y_test, probs.max(1)) if probs is not None else np.nan}
    row.update({f"recall {c}": float((pred[y_test == i] == i).mean()) for i, c in enumerate(CLASS_NAMES)})
    return row


rows, preds = [], {}
for name in LOCAL:
    recs = load_cache(CACHE_DIR / f"{name}_test.jsonl")
    if len(recs) < len(test_df):
        print(f"skipping {name}: {len(recs)}/{len(test_df)} predictions"); continue
    probs = np.array([recs[u]["probs"] for u in test_df.uid])
    lat = np.array([recs[u]["latency_s"] for u in test_df.uid])
    mem_f = CACHE_DIR / f"{name}_test_mem.json"
    preds[name] = {"probs": probs, "pred": probs.argmax(1), "latency": lat}
    gen = [(recs[u].get("generated"), probs[i].argmax()) for i, u in enumerate(test_df.uid) if "generated" in recs[u]]
    rows.append({"model": name, **metrics(probs.argmax(1), 1 - probs[:, SAFE] >= 0.5, probs),
                 "ms / image": 1000 * float(np.median(lat)),
                 "peak GPU GB": json.loads(mem_f.read_text())["peak_gb"] if mem_f.exists() else np.nan,
                 "$ / 1k images": 1000 * float(np.median(lat)) / 3600 * T4_USD_PER_HOUR,
                 "generated = top letter": (np.mean([g[:1].upper() == LETTERS[t] for g, t in gen if g]) if gen else np.nan)})

claude = load_cache(CACHE_DIR / "claude_test.jsonl")
if len(claude) == len(test_df):
    cats = [claude[u]["category"] for u in test_df.uid]
    pred = np.array([CLASS_NAMES.index(c) if c else -1 for c in cats])
    usd = np.mean([(claude[u]["input_tokens"] * CLAUDE_USD_PER_MTOK["input"]
                    + claude[u]["output_tokens"] * CLAUDE_USD_PER_MTOK["output"]) / 1e6 for u in test_df.uid])
    preds[CLAUDE_NAME] = {"pred": pred, "latency": np.array([claude[u]["latency_s"] for u in test_df.uid])}
    rows.append({"model": CLAUDE_NAME, **metrics(pred, (pred != SAFE) & (pred >= 0)),
                 "ms / image": 1000 * float(np.median(preds[CLAUDE_NAME]["latency"])), "$ / 1k images": 1000 * usd,
                 "refusals": int((pred < 0).sum()),
                 "served by": ", ".join(sorted({claude[u]["served_by"] for u in test_df.uid}))})
elif claude:
    print(f"Claude: only {len(claude)}/{len(test_df)} predictions, left out of the tables")

results = pd.DataFrame(rows).set_index("model")
results.to_csv("summary.csv")
pct = ["accuracy", "lenient accuracy", "hazard recall", "false-alarm rate", "precision"] + [f"recall {c}" for c in CLASS_NAMES]
display(results.style.format({**{c: "{:.1%}" for c in pct}, "macro-F1": "{:.3f}", "ECE": "{:.3f}", "ms / image": "{:.0f}",
                              "peak GPU GB": "{:.1f}", "$ / 1k images": "${:.2f}", "generated = top letter": "{:.0%}"},
                             na_rep=""))

In [ ]:
# Per-image predictions for the repo (no images), and exact McNemar tests on paired correctness.
per_image = test_df[["uid", "category"]].copy()
for name, p in preds.items():
    per_image[name] = [CLASS_NAMES[i] if i >= 0 else "" for i in p["pred"]]
per_image.to_csv("results.csv", index=False)


def mcnemar(a, b):
    ca, cb = preds[a]["pred"] == y_test, preds[b]["pred"] == y_test
    only_a, only_b = int((ca & ~cb).sum()), int((~ca & cb).sum())
    p = binomtest(only_a, only_a + only_b, 0.5).pvalue if only_a + only_b else 1.0
    return {"A": a, "B": b, "only A right": only_a, "only B right": only_b, "p": round(p, 4)}


best_local = results.loc[[m for m in LOCAL if m in results.index], "accuracy"].idxmax() if len(results) else None
pairs = [("d1-3B", "LFM2.5-VL-3B"), ("d1-3B", "Qwen3.5-4B"), ("d1-3B", FT_NAME), (best_local, CLAUDE_NAME)]
display(pd.DataFrame([mcnemar(a, b) for a, b in pairs if a in preds and b in preds and a != b]))

## "Needs attention" threshold chosen on the calibration split

In practice you'd flag an image for a person to check when P(hazard) passes a threshold. The rule is fixed in advance: for each local model, take the **highest threshold that catches at least 90% of the hazard images in the 100-image calibration split**, then apply it unchanged to the test set. The test set plays no part in the choice. The calibration split has about 70 hazard images, so the 90% target isn't set by a single image. Claude is shown at its single operating point.

In [ ]:
def pick_threshold(hz, p_hz, target):
    # Highest threshold whose hazard recall on these images is at least `target`.
    for t in np.unique(p_hz)[::-1]:
        if (p_hz[hz] >= t).mean() >= target:
            return float(t)
    return 0.0


hz_calib = calib_df.label.to_numpy() != SAFE
thr_rows, p_hz = [], {}
for name in [m for m in LOCAL if m in preds]:
    cal = load_cache(CACHE_DIR / f"{name}_calib.jsonl")
    if len(cal) < len(calib_df):
        print(f"skipping {name}: calibration predictions missing"); continue
    pc = 1 - np.array([cal[u]["probs"][SAFE] for u in calib_df.uid])
    pt = 1 - preds[name]["probs"][:, SAFE]
    p_hz[name] = pt
    t = pick_threshold(hz_calib, pc, HAZARD_RECALL_TARGET)
    for rule, thr in [("default 0.5", 0.5), (f"calib target {HAZARD_RECALL_TARGET:.0%}", t)]:
        thr_rows.append({"model": name, "rule": rule, "threshold": round(thr, 4),
                         "calib hazard recall": float((pc[hz_calib] >= thr).mean()), **attention(pt >= thr, hz_test)})
if CLAUDE_NAME in preds:
    pr = preds[CLAUDE_NAME]["pred"]
    thr_rows.append({"model": CLAUDE_NAME, "rule": "label only", **attention((pr != SAFE) & (pr >= 0), hz_test)})
thresholds = pd.DataFrame(thr_rows)
thresholds.to_csv("thresholds.csv", index=False)
display(thresholds.style.format({"threshold": "{:.3f}", "calib hazard recall": "{:.0%}", "hazard recall": "{:.0%}",
                                 "false-alarm rate": "{:.0%}", "precision": "{:.0%}"}, na_rep=""))

## Results

In [ ]:
palette = dict(zip(LOCAL + [CLAUDE_NAME], sns.color_palette("tab10", len(LOCAL) + 1)))
r = results.copy()
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Accuracy vs cost per 1,000 images (log scale)
ax = axes[0]
for name, row in r.iterrows():
    ax.scatter(row["$ / 1k images"], row["accuracy"], s=70, color=palette.get(name, "grey"), zorder=3)
    ax.annotate(name, (row["$ / 1k images"], row["accuracy"]), textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("cost per 1,000 images, USD (log scale)"); ax.set_ylabel("accuracy (5-way)")
ax.set_title("Accuracy vs cost"); ax.set_ylim(0, 1.02)
ax.text(0.01, 0.01, f"Local models: one image at a time on a T4 at ${T4_USD_PER_HOUR}/hour (assumption).",
        transform=ax.transAxes, fontsize=7, color="#777777")

# Hazard recall vs false alarms across thresholds (test set; for display only)
ax = axes[1]
n_hz, n_safe = int(hz_test.sum()), int((~hz_test).sum())
for name, pt in p_hz.items():
    order = np.argsort(-pt)
    caught = np.r_[0, np.cumsum(hz_test[order])] / n_hz
    alarms = np.r_[0, np.cumsum(~hz_test[order])] / n_safe
    ax.plot(alarms, caught, color=palette[name], lw=1.4, label=name)
    for _, t in thresholds[thresholds.model == name].iterrows():
        hollow = t.rule.startswith("default")
        ax.scatter(t["false-alarm rate"], t["hazard recall"], s=55, marker="o" if hollow else "^", zorder=4,
                   color="white" if hollow else palette[name], edgecolors=[palette[name]], linewidths=1.6)
if CLAUDE_NAME in preds:
    t = thresholds[thresholds.model == CLAUDE_NAME].iloc[0]
    ax.scatter(t["false-alarm rate"], t["hazard recall"], marker="s", s=60, color=palette[CLAUDE_NAME], zorder=4,
               label=f"{CLAUDE_NAME} (label)")
ax.scatter([], [], s=55, marker="o", color="white", edgecolors="#555555", label="threshold 0.5")
ax.scatter([], [], s=55, marker="^", color="#555555", label=f"calibration target {HAZARD_RECALL_TARGET:.0%}")
ax.set_xlabel(f"false-alarm rate (of {n_safe} safe images)"); ax.set_ylabel(f"hazard recall (of {n_hz} hazard images)")
ax.set_title("Needs attention: hazards caught vs false alarms"); ax.set_xlim(-0.02, 1.02); ax.set_ylim(0, 1.02)
ax.legend(fontsize=7, frameon=False, loc="lower right")
plt.tight_layout(); plt.savefig(FIG_DIR / "accuracy_cost_attention.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
show_cm = [m for m in ["d1-3B", "LFM2.5-VL-3B", FT_NAME, CLAUDE_NAME] if m in preds]
fig, axes = plt.subplots(1, len(show_cm), figsize=(4.6 * len(show_cm), 4.4))
short = ["spill", "forklift", "stacking", "obstacle", "safe"]
for ax, m in zip(np.atleast_1d(axes), show_cm):
    ok = preds[m]["pred"] >= 0
    cm = confusion_matrix(y_test[ok], preds[m]["pred"][ok], labels=range(5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax, xticklabels=short, yticklabels=short)
    ax.set_title(m, fontsize=10); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.savefig(FIG_DIR / "confusion.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Calibration of P(hazard): grouped by predicted probability, against the actual share of hazard images.
BINS = [0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0001]
fig, axes = plt.subplots(1, len(p_hz), figsize=(3.4 * len(p_hz), 3.6), sharey=True)
rel_rows = []
for ax, (name, pt) in zip(np.atleast_1d(axes), p_hz.items()):
    which = np.digitize(pt, BINS) - 1
    g = [(pt[which == b].mean(), hz_test[which == b].mean(), int((which == b).sum())) for b in range(len(BINS) - 1)
         if (which == b).any()]
    rel_rows += [{"model": name, "mean P(hazard)": a, "actual hazard share": h, "images": n} for a, h, n in g]
    ax.plot([0, 1], [0, 1], color="#bbbbbb", ls="--", lw=1)
    ax.plot([a for a, _, _ in g], [h for _, h, _ in g], color=palette[name], lw=1.2)
    ax.scatter([a for a, _, _ in g], [h for _, h, _ in g], s=[12 + 2 * n for _, _, n in g], color=palette[name], zorder=3)
    ax.set_title(name, fontsize=9); ax.set_xlabel("mean P(hazard)"); ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.03, 1.03)
np.atleast_1d(axes)[0].set_ylabel("actual share of hazard images")
plt.tight_layout(); plt.savefig(FIG_DIR / "reliability_hazard.png", dpi=150, bbox_inches="tight"); plt.show()
pd.DataFrame(rel_rows).to_csv("reliability_hazard.csv", index=False)

## Qualitative examples

Six test images: hazards that d1-3B missed, images where the models disagree, and one that all of them got right.

In [ ]:
show = [m for m in ["d1-3B", "LFM2.5-VL-3B", FT_NAME, CLAUDE_NAME] if m in preds]
if show:
    t = test_df.assign(**{m: preds[m]["pred"] for m in show})
    missed = t[(t.label != SAFE) & (t[show[0]] == SAFE)]
    disagree = t[t[show].nunique(axis=1) > 1]
    all_right = t[(t[show].values == t.label.values[:, None]).all(1)]
    pick = pd.concat([missed.sample(min(2, len(missed)), random_state=SEED),
                      disagree.sample(min(3, len(disagree)), random_state=SEED),
                      all_right.sample(min(1, len(all_right)), random_state=SEED)]).drop_duplicates("uid").head(6)
    fig, axes = plt.subplots(1, len(pick), figsize=(4.2 * len(pick), 4.6))
    for ax, (_, row) in zip(np.atleast_1d(axes), pick.iterrows()):
        ax.imshow(IMAGES[row.uid]); ax.axis("off")
        lines = [f"true: {row.category}"] + [f"{m}: {CLASS_NAMES[row[m]] if row[m] >= 0 else 'refused'}" for m in show]
        ax.set_title("\n".join(lines), fontsize=7.5, loc="left")
    plt.tight_layout(); plt.savefig(FIG_DIR / "examples_predictions.png", dpi=130, bbox_inches="tight"); plt.show()

## Interpretation

TODO(you): fill this in after running the full notebook.
- Does the decision post-training help? Compare d1-3B with its own base, LFM2.5-VL-3B (McNemar), on accuracy, hazard recall and calibration.
- Did our zero-shot Qwen3.5-2B and 4B land near the dataset card's 67.2% and 83.3%? If not, why might our prompt differ?
- How close does the fine-tuned 0.8B model get to Claude, and at what cost per 1,000 images?
- On "needs attention": with the calibration-chosen threshold, how many hazards does each model miss, and how many false alarms does it raise?
- Which hazard classes are hardest (per-class recall), and for which models?

## Limitations & next steps

- **Synthetic images.** Every image was rendered and edited with Gemini, so results may not transfer to real cameras.
- **Small test set.** There are 186 images, only 21–48 per class, so per-class numbers are noisy.
- **Prompt sensitivity.** The generative VLMs are scored on next-token letter probabilities with one prompt. Other prompts or letter orders could shift results.
- **Cost model.** Local costs assume one image at a time on a T4 at a fixed hourly price. Batching would cut them substantially. Claude's cost uses measured tokens at list price.
- **Fine-tuning path.** The fine-tuned arm is a vision LoRA, not an Unsloth decision head, because image decision training isn't available in Unsloth 2026.10.3.
- **Next steps:** real CCTV imagery, batched inference costs, fine-tuning d1 itself if supported, and a cost-weighted threshold.

## Environment

In [ ]:
!pip freeze | grep -iE "^(unsloth|unsloth.zoo|transformers|datasets|torch|peft|trl|bitsandbytes|accelerate|anthropic|scikit-learn)=="

## Download all results

This cell zips the figures, the CSVs, the prediction caches (image ids, probabilities and Claude token usage; no images) and the fine-tuning summary, and downloads the zip on Colab. The model adapter is left out.

In [ ]:
import zipfile
from datetime import datetime

zip_path = Path(f"warehouse_results_{datetime.now():%Y%m%d_%H%M}.zip")
files = [*FIG_DIR.glob("*.png"), *(Path(f) for f in ["summary.csv", "results.csv", "thresholds.csv", "reliability_hazard.csv"])]
files += [*CACHE_DIR.glob("*.jsonl"), *CACHE_DIR.glob("*_mem.json"), *CKPT_DIR.glob("*/done.json")]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if f.exists():
            z.write(f, f.relative_to(BASE_DIR) if BASE_DIR in f.parents else f)
print(f"{zip_path}: {len(z.namelist())} files, {zip_path.stat().st_size / 1e6:.1f} MB")
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except ImportError:
    print("Not on Colab: the zip is next to the notebook.")